# SmartSentry AML — Pipeline Orchestrator

**One-click sequential execution of the entire AML pipeline.**

This notebook runs all 6 modules in order, validates outputs between stages, and produces a final summary report. Each module is executed as a subprocess using `nbconvert`, so kernel state is isolated and any single-module failure is caught without crashing the entire pipeline.

### Pipeline Order
```
[00] aml_generator_complete_pipeline.ipynb   → Synthetic data generation
[01] 01__aml_typology_detector.ipynb         → Graph-based typology detection
[02] 02__aml_rules_engine.ipynb              → 126-rule compliance engine
[03] 03__aml_feature_engineering.ipynb        → Velocity/balance/IP features
[04] 04__aml_ml_preparation.ipynb            → Phase 1: Binary AML detection
[05] 05__aml_phase2_typology_classifier.ipynb → Phase 2: Typology classification
```


## 1 — Setup & Configuration


In [ ]:
import os
import sys
import time
import json
import subprocess
from datetime import datetime, timedelta
from pathlib import Path

# ═══════════════════════════════════════════════════════════════
# CONFIGURATION — Update these paths to match your environment
# ═══════════════════════════════════════════════════════════════

# Directory containing all notebooks
NOTEBOOK_DIR = os.getcwd()

# Output directory (parent-level outputs_updated)
OUTPUT_DIR = os.path.join(os.path.dirname(NOTEBOOK_DIR), "outputs_updated")

# Pipeline notebooks in execution order
PIPELINE = [
    {
        "id": "00",
        "name": "Data Generator",
        "notebook": "aml_generator_complete_pipeline.ipynb",
        "outputs": ["transactions_generated_typology_V2.parquet"],
        "description": "Generate synthetic transactions with 10 AML typologies",
    },
    {
        "id": "01",
        "name": "Typology Detector",
        "notebook": "01__aml_typology_detector.ipynb",
        "outputs": ["stg_transactions_flagged.parquet"],
        "description": "Graph-based detection of AML patterns, assign is_aml labels",
    },
    {
        "id": "02",
        "name": "Rules Engine",
        "notebook": "02__aml_rules_engine.ipynb",
        "outputs": ["stg_transactions_rules_V2.parquet"],
        "description": "Apply 126 regulatory compliance rules (RBI/PMLA/FIU-IND)",
    },
    {
        "id": "03",
        "name": "Feature Engineering",
        "notebook": "03__aml_feature_engineering.ipynb",
        "outputs": ["stg_transactions_features_V2.parquet"],
        "description": "Compute velocity, balance, IP risk, and volume features",
    },
    {
        "id": "04",
        "name": "Phase 1: AML Detection",
        "notebook": "04__aml_ml_preparation.ipynb",
        "outputs": [
            "ml_outputs/final_lgb_model.txt",
            "ml_outputs/model_metadata.json",
            "ml_outputs/X_train.parquet",
            "ml_outputs/X_test.parquet",
            "ml_outputs/y_train.parquet",
            "ml_outputs/y_test.parquet",
        ],
        "description": "Train binary AML classifier with hyperparameter tuning",
    },
    {
        "id": "05",
        "name": "Phase 2: Typology Classifier",
        "notebook": "05__aml_phase2_typology_classifier.ipynb",
        "outputs": [
            "phase2_outputs/phase2_typology_model.txt",
            "phase2_outputs/phase2_metadata.json",
            "phase2_outputs/combined_aml_output.parquet",
        ],
        "description": "Train 10-class typology classifier with multi-label threshold",
    },
]

# Execution settings
TIMEOUT_MINUTES = 60          # Max time per notebook (increase for large datasets)
STOP_ON_FAILURE = True        # Stop pipeline if any notebook fails
SAVE_EXECUTED_NOTEBOOKS = True  # Save executed notebooks with outputs

print("=" * 70)
print("SmartSentry AML — Pipeline Orchestrator")
print("=" * 70)
print(f"  Notebook directory:  {NOTEBOOK_DIR}")
print(f"  Output directory:    {OUTPUT_DIR}")
print(f"  Timeout per module:  {TIMEOUT_MINUTES} minutes")
print(f"  Stop on failure:     {STOP_ON_FAILURE}")
print(f"  Modules to run:      {len(PIPELINE)}")
print()

# Verify all notebooks exist
all_found = True
for stage in PIPELINE:
    nb_path = os.path.join(NOTEBOOK_DIR, stage["notebook"])
    exists = os.path.exists(nb_path)
    status = "✓" if exists else "⚠ NOT FOUND"
    print(f"  [{stage['id']}] {stage['notebook']:<55s} {status}")
    if not exists:
        all_found = False

if not all_found:
    raise FileNotFoundError("One or more notebooks are missing. Fix paths above.")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\n  ✓ All notebooks found. Ready to execute.")


## 2 — Notebook Runner Engine


In [ ]:
def run_notebook(notebook_path, timeout_minutes=60, working_dir=None):
    """
    Execute a Jupyter notebook using nbconvert and return status.
    The notebook runs in its own kernel (isolated state).
    """
    start_time = time.time()
    notebook_name = os.path.basename(notebook_path)
    
    # Output path for executed notebook (with cell outputs preserved)
    executed_dir = os.path.join(OUTPUT_DIR, "executed_notebooks")
    os.makedirs(executed_dir, exist_ok=True)
    executed_path = os.path.join(executed_dir, notebook_name)
    
    print(f"  Executing: {notebook_name}")
    print(f"  Working dir: {working_dir or os.path.dirname(notebook_path)}")
    
    try:
        cmd = [
            sys.executable, "-m", "jupyter", "nbconvert",
            "--to", "notebook",
            "--execute",
            "--ExecutePreprocessor.timeout=" + str(timeout_minutes * 60),
            "--ExecutePreprocessor.kernel_name=python3",
            "--output", executed_path,
            notebook_path,
        ]
        
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=timeout_minutes * 60 + 60,  # Extra buffer for nbconvert overhead
            cwd=working_dir or os.path.dirname(notebook_path),
        )
        
        elapsed = time.time() - start_time
        
        if result.returncode == 0:
            return {
                "status": "SUCCESS",
                "elapsed_seconds": elapsed,
                "elapsed_str": str(timedelta(seconds=int(elapsed))),
                "executed_path": executed_path if SAVE_EXECUTED_NOTEBOOKS else None,
                "stdout": result.stdout[-500:] if result.stdout else "",
                "stderr": "",
            }
        else:
            # Extract error from stderr
            error_msg = result.stderr[-1000:] if result.stderr else "Unknown error"
            return {
                "status": "FAILED",
                "elapsed_seconds": elapsed,
                "elapsed_str": str(timedelta(seconds=int(elapsed))),
                "error": error_msg,
                "stdout": result.stdout[-500:] if result.stdout else "",
                "stderr": result.stderr[-500:] if result.stderr else "",
            }
    
    except subprocess.TimeoutExpired:
        elapsed = time.time() - start_time
        return {
            "status": "TIMEOUT",
            "elapsed_seconds": elapsed,
            "elapsed_str": str(timedelta(seconds=int(elapsed))),
            "error": f"Notebook exceeded {timeout_minutes} minute timeout",
        }
    except Exception as e:
        elapsed = time.time() - start_time
        return {
            "status": "ERROR",
            "elapsed_seconds": elapsed,
            "elapsed_str": str(timedelta(seconds=int(elapsed))),
            "error": str(e),
        }


def validate_outputs(stage, output_dir):
    """Check that expected output files were created by a stage."""
    results = []
    for output_file in stage["outputs"]:
        full_path = os.path.join(output_dir, output_file)
        if os.path.exists(full_path):
            size_mb = os.path.getsize(full_path) / (1024 * 1024)
            results.append({"file": output_file, "status": "✓", "size_mb": size_mb})
        else:
            results.append({"file": output_file, "status": "⚠ MISSING", "size_mb": 0})
    return results


print("Runner engine loaded.")
print("  run_notebook()     — executes a notebook via nbconvert")
print("  validate_outputs() — checks expected output files exist")


## 3 — Execute Full Pipeline

This cell runs all 6 notebooks sequentially. Each notebook:
1. Executes in its own isolated kernel
2. Has its outputs validated after completion
3. Logs timing and status

**Estimated total runtime:** 15–45 minutes depending on hardware.


In [ ]:
print("=" * 70)
print("PIPELINE EXECUTION STARTED")
print(f"  Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 70)

pipeline_start = time.time()
pipeline_results = []
pipeline_failed = False

for i, stage in enumerate(PIPELINE):
    print(f"\n{'─' * 70}")
    print(f"  STAGE [{stage['id']}] {stage['name']}")
    print(f"  {stage['description']}")
    print(f"{'─' * 70}")
    
    if pipeline_failed and STOP_ON_FAILURE:
        print(f"  ⊘ SKIPPED (previous stage failed)")
        pipeline_results.append({
            "stage": stage["id"],
            "name": stage["name"],
            "status": "SKIPPED",
            "elapsed_str": "—",
            "elapsed_seconds": 0,
        })
        continue
    
    # Execute notebook
    nb_path = os.path.join(NOTEBOOK_DIR, stage["notebook"])
    result = run_notebook(nb_path, timeout_minutes=TIMEOUT_MINUTES, working_dir=NOTEBOOK_DIR)
    
    # Status indicator
    status_icon = {"SUCCESS": "✓", "FAILED": "✗", "TIMEOUT": "⏱", "ERROR": "⚠"}.get(result["status"], "?")
    print(f"\n  {status_icon} Status: {result['status']} ({result['elapsed_str']})")
    
    if result["status"] != "SUCCESS":
        print(f"  Error: {result.get('error', 'Unknown')[:300]}")
        if result.get("stderr"):
            print(f"  Stderr: {result['stderr'][:300]}")
        pipeline_failed = True
    
    # Validate outputs
    if result["status"] == "SUCCESS":
        validations = validate_outputs(stage, OUTPUT_DIR)
        print(f"\n  Output Validation:")
        all_valid = True
        for v in validations:
            print(f"    {v['status']} {v['file']:<50s} {v['size_mb']:>8.2f} MB")
            if v["status"] != "✓":
                all_valid = False
        
        if not all_valid:
            print(f"\n  ⚠ WARNING: Some expected outputs are missing.")
            print(f"    Pipeline will continue but downstream modules may fail.")
    
    pipeline_results.append({
        "stage": stage["id"],
        "name": stage["name"],
        "notebook": stage["notebook"],
        "status": result["status"],
        "elapsed_str": result["elapsed_str"],
        "elapsed_seconds": result["elapsed_seconds"],
    })

pipeline_elapsed = time.time() - pipeline_start
print(f"\n{'=' * 70}")
print(f"PIPELINE EXECUTION COMPLETE")
print(f"  Total time: {str(timedelta(seconds=int(pipeline_elapsed)))}")
print(f"  End time:   {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'=' * 70}")


## 4 — Execution Summary


In [ ]:
print("\n" + "=" * 70)
print("PIPELINE SUMMARY REPORT")
print("=" * 70)

print(f"\n  {'Stage':<6s} {'Module':<35s} {'Status':<10s} {'Time':>10s}")
print(f"  {'─' * 65}")

total_success = 0
total_failed = 0
total_skipped = 0

for r in pipeline_results:
    icon = {"SUCCESS":"✓","FAILED":"✗","TIMEOUT":"⏱","SKIPPED":"⊘","ERROR":"⚠"}.get(r["status"],"?")
    print(f"  [{r['stage']}]  {r['name']:<35s} {icon} {r['status']:<8s} {r['elapsed_str']:>10s}")
    
    if r["status"] == "SUCCESS": total_success += 1
    elif r["status"] == "SKIPPED": total_skipped += 1
    else: total_failed += 1

print(f"\n  Total: {total_success} succeeded | {total_failed} failed | {total_skipped} skipped")
print(f"  Pipeline time: {str(timedelta(seconds=int(pipeline_elapsed)))}")

# Overall status
if total_failed == 0 and total_skipped == 0:
    print(f"\n  ✓ PIPELINE COMPLETED SUCCESSFULLY")
elif total_failed > 0:
    print(f"\n  ✗ PIPELINE FAILED — check error logs above")
    failed_stages = [r for r in pipeline_results if r["status"] in ("FAILED", "ERROR", "TIMEOUT")]
    for r in failed_stages:
        print(f"    → Stage [{r['stage']}] {r['name']}: {r['status']}")

# ═══ Output file inventory ═══
print(f"\n{'─' * 70}")
print(f"OUTPUT FILE INVENTORY")
print(f"{'─' * 70}")

total_size = 0
file_count = 0
for root, dirs, files in os.walk(OUTPUT_DIR):
    # Skip executed_notebooks directory for cleaner output
    if "executed_notebooks" in root:
        continue
    for fname in sorted(files):
        fpath = os.path.join(root, fname)
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        rel_path = os.path.relpath(fpath, OUTPUT_DIR)
        print(f"  {rel_path:<60s} {size_mb:>8.2f} MB")
        total_size += size_mb
        file_count += 1

print(f"\n  Total: {file_count} files, {total_size:.2f} MB")
print(f"  Location: {OUTPUT_DIR}")


## 5 — Quick Output Validation

Loads key output files and prints summary statistics to confirm the pipeline produced valid results.


In [ ]:
import pandas as pd

print("\n" + "=" * 70)
print("QUICK VALIDATION")
print("=" * 70)

validation_checks = []

# Check 1: Generator output
gen_file = os.path.join(OUTPUT_DIR, "transactions_generated_typology_V2.parquet")
if os.path.exists(gen_file):
    df_gen = pd.read_parquet(gen_file)
    aml_count = (df_gen.get("is_aml", pd.Series()) == 1).sum()
    total = len(df_gen)
    fraud_rate = aml_count / total * 100 if total > 0 else 0
    print(f"\n  [00] Generator: {total:,} transactions, {aml_count:,} AML ({fraud_rate:.1f}%)")
    validation_checks.append(("Generator", total > 300000 and 15 < fraud_rate < 30))
    del df_gen
else:
    print(f"\n  [00] Generator: ⚠ Output not found")
    validation_checks.append(("Generator", False))

# Check 2: Detector output
det_file = os.path.join(OUTPUT_DIR, "stg_transactions_flagged.parquet")
if os.path.exists(det_file):
    df_det = pd.read_parquet(det_file)
    flagged = (df_det.get("is_aml", pd.Series()) == 1).sum()
    typs = df_det.get("aml_typology", pd.Series()).nunique()
    multi = df_det.get("aml_typology", pd.Series()).astype(str).str.contains(";", na=False).sum()
    print(f"  [01] Detector:  {flagged:,} flagged, {typs} typologies, {multi} multi-label (should be 0)")
    validation_checks.append(("Detector", flagged > 50000 and multi == 0))
    del df_det
else:
    print(f"  [01] Detector:  ⚠ Output not found")
    validation_checks.append(("Detector", False))

# Check 3: Rules output
rules_file = os.path.join(OUTPUT_DIR, "stg_transactions_rules_V2.parquet")
if os.path.exists(rules_file):
    df_rules = pd.read_parquet(rules_file)
    rule_cols = [c for c in df_rules.columns if c.startswith("rule_") and c not in {"rule_score","rules_triggered","rules_triggered_count"}]
    trigger_rate = (df_rules[rule_cols].sum(axis=1) > 0).mean() * 100 if rule_cols else 0
    print(f"  [02] Rules:     {len(rule_cols)} rules, {trigger_rate:.1f}% trigger rate (target: 50-60%)")
    validation_checks.append(("Rules", 40 < trigger_rate < 75))
    del df_rules
else:
    print(f"  [02] Rules:     ⚠ Output not found")
    validation_checks.append(("Rules", False))

# Check 4: Features output
feat_file = os.path.join(OUTPUT_DIR, "stg_transactions_features_V2.parquet")
if os.path.exists(feat_file):
    df_feat = pd.read_parquet(feat_file)
    n_features = len(df_feat.columns)
    print(f"  [03] Features:  {len(df_feat):,} rows × {n_features} columns")
    validation_checks.append(("Features", n_features > 150))
    del df_feat
else:
    print(f"  [03] Features:  ⚠ Output not found")
    validation_checks.append(("Features", False))

# Check 5: Phase 1 model
model_file = os.path.join(OUTPUT_DIR, "ml_outputs", "model_metadata.json")
if os.path.exists(model_file):
    with open(model_file) as f:
        meta = json.load(f)
    print(f"  [04] Phase 1:   AUC={meta.get('auc_roc','?'):.4f}, F1={meta.get('f1_score','?'):.4f}, "
          f"Threshold={meta.get('optimal_threshold','?')}, Features={meta.get('n_features','?')}")
    auc = meta.get("auc_roc", 0)
    validation_checks.append(("Phase 1", auc > 0.90))
else:
    print(f"  [04] Phase 1:   ⚠ Model metadata not found")
    validation_checks.append(("Phase 1", False))

# Check 6: Phase 2 model
p2_meta_file = os.path.join(OUTPUT_DIR, "phase2_outputs", "phase2_metadata.json")
if os.path.exists(p2_meta_file):
    with open(p2_meta_file) as f:
        p2_meta = json.load(f)
    print(f"  [05] Phase 2:   Accuracy={p2_meta.get('phase2_accuracy','?'):.4f}, "
          f"Classes={p2_meta.get('n_classes','?')}, Features={p2_meta.get('n_features','?')}")
    p2_acc = p2_meta.get("phase2_accuracy", 0)
    validation_checks.append(("Phase 2", p2_acc > 0.70))
else:
    print(f"  [05] Phase 2:   ⚠ Model metadata not found")
    validation_checks.append(("Phase 2", False))

# Combined output check
combined_file = os.path.join(OUTPUT_DIR, "phase2_outputs", "combined_aml_output.parquet")
if os.path.exists(combined_file):
    df_combined = pd.read_parquet(combined_file)
    aml_alerts = (df_combined.get("predicted_typology", pd.Series()) != "None").sum()
    multi_label = (df_combined.get("num_typologies_matched", pd.Series(0)) >= 2).sum()
    print(f"\n  Combined Output: {len(df_combined):,} rows, {aml_alerts:,} AML alerts, {multi_label:,} multi-label")
    
    # Priority breakdown
    if "investigation_priority" in df_combined.columns:
        print(f"  Priority: ", end="")
        for pri in ["Critical", "High", "Medium", "Low"]:
            cnt = (df_combined["investigation_priority"] == pri).sum()
            print(f"{pri}={cnt:,} ", end="")
        print()
    del df_combined
else:
    print(f"\n  Combined Output: ⚠ Not found")

# Final verdict
print(f"\n{'─' * 70}")
print(f"VALIDATION SUMMARY")
print(f"{'─' * 70}")
all_passed = True
for name, passed in validation_checks:
    icon = "✓" if passed else "✗"
    print(f"  {icon} {name}")
    if not passed: all_passed = False

if all_passed:
    print(f"\n  ✓ ALL VALIDATIONS PASSED — Pipeline output is ready for deployment")
else:
    print(f"\n  ⚠ SOME VALIDATIONS FAILED — Review output above for details")


## 6 — Run Individual Modules (Optional)

Use this cell to re-run a single module without executing the full pipeline. Useful for debugging or re-running a specific stage after fixing an issue.

**Change `MODULE_TO_RUN`** to the module ID you want to execute (00–05).


In [ ]:
# ═══ Change this to run a specific module ═══
MODULE_TO_RUN = "01"  # Options: "00", "01", "02", "03", "04", "05"

# Find the module
target = next((s for s in PIPELINE if s["id"] == MODULE_TO_RUN), None)
if not target:
    print(f"Module {MODULE_TO_RUN} not found. Valid IDs: {[s['id'] for s in PIPELINE]}")
else:
    print(f"Running single module: [{target['id']}] {target['name']}")
    print(f"  {target['description']}")
    print(f"  Notebook: {target['notebook']}")
    print()
    
    nb_path = os.path.join(NOTEBOOK_DIR, target["notebook"])
    result = run_notebook(nb_path, timeout_minutes=TIMEOUT_MINUTES, working_dir=NOTEBOOK_DIR)
    
    icon = {"SUCCESS":"✓","FAILED":"✗","TIMEOUT":"⏱","ERROR":"⚠"}.get(result["status"],"?")
    print(f"\n  {icon} {result['status']} ({result['elapsed_str']})")
    
    if result["status"] == "SUCCESS":
        validations = validate_outputs(target, OUTPUT_DIR)
        print(f"\n  Outputs:")
        for v in validations:
            print(f"    {v['status']} {v['file']:<50s} {v['size_mb']:>8.2f} MB")
    else:
        print(f"  Error: {result.get('error', 'Unknown')[:500]}")
        if result.get("stderr"):
            print(f"\n  Stderr (last 500 chars):")
            print(f"  {result['stderr'][:500]}")


## 7 — Run Pipeline from a Specific Stage (Optional)

If a module failed and you've fixed it, use this to resume from that stage instead of re-running the entire pipeline. All subsequent modules will also be re-run.


In [ ]:
# ═══ Change this to the stage to START from ═══
START_FROM = "02"  # Will run 02, 03, 04, 05

print(f"Running pipeline from stage [{START_FROM}] onwards...")
print()

start_idx = next((i for i, s in enumerate(PIPELINE) if s["id"] == START_FROM), None)
if start_idx is None:
    print(f"Stage {START_FROM} not found. Valid IDs: {[s['id'] for s in PIPELINE]}")
else:
    stages_to_run = PIPELINE[start_idx:]
    print(f"  Stages to execute: {[s['id'] + ' ' + s['name'] for s in stages_to_run]}")
    print()
    
    partial_results = []
    failed = False
    partial_start = time.time()
    
    for stage in stages_to_run:
        print(f"{'─' * 50}")
        print(f"  [{stage['id']}] {stage['name']}")
        
        if failed and STOP_ON_FAILURE:
            print(f"  ⊘ SKIPPED")
            partial_results.append({"stage": stage["id"], "name": stage["name"], "status": "SKIPPED", "elapsed_str": "—"})
            continue
        
        nb_path = os.path.join(NOTEBOOK_DIR, stage["notebook"])
        result = run_notebook(nb_path, timeout_minutes=TIMEOUT_MINUTES, working_dir=NOTEBOOK_DIR)
        
        icon = {"SUCCESS":"✓","FAILED":"✗","TIMEOUT":"⏱","ERROR":"⚠"}.get(result["status"],"?")
        print(f"  {icon} {result['status']} ({result['elapsed_str']})")
        
        if result["status"] != "SUCCESS":
            print(f"  Error: {result.get('error', 'Unknown')[:300]}")
            failed = True
        else:
            validations = validate_outputs(stage, OUTPUT_DIR)
            for v in validations:
                print(f"    {v['status']} {v['file']}")
        
        partial_results.append({"stage": stage["id"], "name": stage["name"], "status": result["status"], "elapsed_str": result["elapsed_str"]})
    
    partial_elapsed = time.time() - partial_start
    print(f"\n{'─' * 50}")
    print(f"  Partial pipeline complete: {str(timedelta(seconds=int(partial_elapsed)))}")
    for r in partial_results:
        icon = {"SUCCESS":"✓","FAILED":"✗","TIMEOUT":"⏱","SKIPPED":"⊘"}.get(r["status"],"?")
        print(f"    {icon} [{r['stage']}] {r['name']}: {r['status']} ({r['elapsed_str']})")


## 8 — Save Execution Log


In [ ]:
# Save pipeline execution log as JSON
log = {
    "pipeline_name": "SmartSentry AML",
    "execution_date": datetime.now().isoformat(),
    "total_elapsed_seconds": pipeline_elapsed,
    "total_elapsed_str": str(timedelta(seconds=int(pipeline_elapsed))),
    "notebook_dir": NOTEBOOK_DIR,
    "output_dir": OUTPUT_DIR,
    "stages": pipeline_results,
    "overall_status": "SUCCESS" if all(r["status"] == "SUCCESS" for r in pipeline_results) else "FAILED",
}

log_path = os.path.join(OUTPUT_DIR, "pipeline_execution_log.json")
with open(log_path, "w") as f:
    json.dump(log, f, indent=2, default=str)

print(f"Execution log saved: {log_path}")
print(f"\n{'=' * 70}")
print(f"ORCHESTRATOR COMPLETE")
print(f"{'=' * 70}")
